# 00 · Start here: concepts, taxonomy, detectors

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/00_start_here.ipynb)

[series index](https://github.com/jstjoe/local-privacy/blob/main/notebooks/README.md) · [01_datasets](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/01_datasets.ipynb) →

This series benchmarks open-weight and hosted **PII detectors** against labelled
datasets, then shows what you can do with what they find: **sanitize** the text,
and still **use** it downstream.

| notebook | what it covers | |
|---|---|---|
| [00_start_here](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/00_start_here.ipynb) | Start here: concepts, taxonomy, detectors | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/00_start_here.ipynb) |
| [01_datasets](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/01_datasets.ipynb) | Datasets: built-in samples and your own data | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/01_datasets.ipynb) |
| [02_run_detectors](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/02_run_detectors.ipynb) | Run detectors over a sample | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/02_run_detectors.ipynb) |
| [03_score_and_compare](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/03_score_and_compare.ipynb) | Score, compare, and inspect errors | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/03_score_and_compare.ipynb) |
| [04_composite_detectors](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/04_composite_detectors.ipynb) | Composite detectors: best-per-category ensembles | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/04_composite_detectors.ipynb) |
| [05_sanitization](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/05_sanitization.ipynb) | Sanitization: redact, label, number, tokenize | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/05_sanitization.ipynb) |
| [06_search_on_sanitized_data](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/06_search_on_sanitized_data.ipynb) | Use: search over sanitized text | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/06_search_on_sanitized_data.ipynb) |

Each notebook runs on its own — later ones create whatever results they need if
you haven't run the earlier ones. A GPU runtime (Colab: *Runtime → Change
runtime type → T4 GPU*) makes the detector runs 5–10× faster but isn't required.

This first notebook needs no downloads beyond the setup cell below. It
introduces the ideas the rest of the series builds on.

In [ ]:
# Setup — run once per session. On Colab this clones and installs the harness
# (a few minutes the first time). Locally, after `uv sync`, it does nothing.
HARNESS_REPO = "https://github.com/jstjoe/local-privacy.git"
HARNESS_REF = "main"   # branch or tag — e.g. a PR branch to try it before merging
OPF_REPO = "https://github.com/openai/privacy-filter.git"
OPF_REF = "main"

import importlib.util, os, subprocess, sys

if importlib.util.find_spec("opf_eval") is None:
    home = "/content" if "google.colab" in sys.modules else os.path.expanduser("~")
    for repo, ref, dest in [(OPF_REPO, OPF_REF, f"{home}/privacy-filter"),
                            (HARNESS_REPO, HARNESS_REF, f"{home}/local-privacy")]:
        if not os.path.exists(dest):
            subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", ref, repo, dest], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{home}/privacy-filter",
                    f"{home}/local-privacy/eval[notebooks]", f"{home}/local-privacy/api"], check=True)
    # A running kernel doesn't always see freshly pip-installed packages;
    # putting the sources on sys.path makes the imports below work either way.
    sys.path[:0] = [f"{home}/privacy-filter", f"{home}/local-privacy/eval/src", f"{home}/local-privacy/api/src"]

from opf_eval import nb
ws = nb.setup()

## Three questions

1. **Detection:** where is the sensitive data? A detector takes text and returns
   *spans*: character offsets plus a label such as `EMAIL` or `GOV_ID`.
2. **Sanitization:** what do you replace it with? Asterisks, a label, a
   per-document number, or a deterministic token each keep a different amount
   of the original's usefulness.
3. **Use:** does the sanitized text still work for search, joins, analytics or
   RAG? That depends on which sanitization you chose.

Notebooks 01–04 measure detection, 05 covers sanitization and 06 covers use.

## One vocabulary for every detector

Every detector speaks its own label language. Presidio says `US_SSN`, OPF says
`account_number`, Skyflow says `SSN`, and a GLiNER model says whatever you
prompted it with. Datasets differ too. To compare them, the harness maps every
raw label into one **canonical taxonomy** with two levels:

- **Coarse:** 15 top-level categories such as `PERSON`, `EMAIL`, `ADDRESS` and
  `ACCOUNT`. Every detector and dataset can be compared at this level.
- **Fine:** sub-types under some categories, for example `GOV_ID`,
  `MEDICAL_ID` and `BANK_ACCOUNT` under `ACCOUNT`, or `GIVEN_NAME` under
  `PERSON`. Newer benchmarks annotate at this level. Scoring at it rewards
  detectors that tell an SSN from a bank account.

In [ ]:
from opf_eval import taxonomy

nb.show_table(
    [
        {
            "coarse": c,
            "fine sub-types": ", ".join(taxonomy.children(c)) or None,
            "meaning": taxonomy.LABELS[c].description,
        }
        for c in taxonomy.COARSE_LABELS
    ]
)

Each source's raw labels map to the most specific canonical label they
unambiguously mean. For example, here is how different sources label a US
Social Security number, and how OPF's catch-all `account_number` stays coarse:

In [ ]:
examples = [
    ("presidio", "US_SSN"),
    ("skyflow", "SSN"),
    ("gretel", "ssn"),
    ("pii300k", "SOCIALNUMBER"),
    ("opf", "account_number"),
    ("pii200k", "IBAN"),
]
nb.show_table([
    {
        "source": src,
        "raw label": raw,
        "fine": taxonomy.to_canonical(src, raw),
        "coarse": taxonomy.to_canonical(src, raw, level="coarse"),
    }
    for src, raw in examples
])

## Detectors

Detectors are registered by name, and every notebook builds them by that name.
Nothing is downloaded until a detector is actually built. `available` shows
whether the libraries it needs are installed in this runtime.

In [ ]:
from opf_eval.detectors import registry

nb.show_table(registry.describe(), ["name", "vocab", "license", "available", "description"])

## Datasets

The built-in datasets are the ai4privacy family on Hugging Face. They're
synthetic, multilingual and span-annotated, and differ in size and label
vocabulary. [01 · Datasets](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/01_datasets.ipynb) shows how to add your own.

In [ ]:
from opf_eval import datasets

nb.show_table(datasets.describe())

## How detectors are scored

A detector's spans are compared with the dataset's gold spans using the
[SemEval 2013](https://aclanthology.org/S13-2056/) entity-matching schemas
(via [`nervaluate`](https://github.com/MantisAI/nervaluate)):

| schema | a prediction counts as correct when… |
|---|---|
| **Strict** | the boundaries match exactly **and** the label matches |
| **Exact** | the boundaries match exactly (label ignored) |
| **Partial** | the spans overlap (label ignored) |
| **Type** | the spans overlap **and** the label matches |

There are two views of the same predictions:

- **Fair:** each detector is scored only on the categories it claims to
  detect, intersected with the categories the dataset annotates.
  Apples-to-apples within each detector's coverage.
- **Raw:** every detector is scored on everything the dataset annotates.
  Categories a detector can't produce count as misses, so this view shows
  out-of-the-box coverage.

Both views can be scored at the coarse or the fine **level**. At the fine
level, a category is split into sub-types only when both the detector and the
dataset distinguish them. Otherwise it's compared at the coarse level, so a
detector isn't penalised for granularity it never claimed.

## Try a detector

Presidio is the lightest detector to start with: rules plus a spaCy model, with
no GPU needed. The next cell installs the spaCy model if it's missing, then
detects PII in one sentence.

In [ ]:
nb.ensure_spacy_model("en_core_web_lg")
presidio = registry.build("presidio")

text = "Hi, I'm Dana Whitfield — reach me at dana.w@example.org or +1 (415) 555-0199. Card on file: 4111 1111 1111 1111."
result = presidio.detect(text)
nb.show_table(result["spans"], ["text", "label", "fine_label", "raw_label", "start", "end"])
print(f"{result['latency_ms']:.1f} ms")

`label` is the coarse category, `fine_label` is the most specific canonical
label, and `raw_label` is what the detector itself returned. Every detector in
the registry returns this same shape.

## Next

Pick a dataset sample to benchmark on, or bring your own data.

**[01_datasets](https://colab.research.google.com/github/jstjoe/local-privacy/blob/main/notebooks/01_datasets.ipynb)** — Datasets: built-in samples and your own data.